# 06 · Image Manipulation for Vision

**Goal:** change one image on purpose, compare the result, and decide whether a vision output should still be trusted.

We use the same public source image for every transformation so that the change is visible and reproducible.


## 1. Get one source image

The image below comes from the Ultralytics public example set. Download it once, then keep the original unchanged.


In [ ]:
# Imports: PIL performs every image transformation in this notebook.
# display() shows images directly in Colab/Jupyter - no matplotlib is used.
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from IPython.display import display

# Keep this URL fixed so every transformation starts from the same source.
IMAGE_URL = 'https://ultralytics.com/images/bus.jpg'
source_path = Path('bus.jpg')

# Download once, then reuse the unchanged source image.
if not source_path.exists():
    urlretrieve(IMAGE_URL, source_path)

original = Image.open(source_path).convert('RGB')
print(f'Original size: {original.size}')


In [ ]:
# Look at the unchanged image before making a prediction.
display(original)


## 2. Make controlled transformations

Each function changes a different kind of evidence. Keep the original as the comparison point.


In [ ]:
# Each function changes one kind of visual evidence.
# Edit ONE parameter at a time when you test a claim.
def crop_context(image, fraction=0.28):
    # fraction controls how much side context is removed.
    width, height = image.size
    return image.crop((width * fraction, height * 0.08, width * (1 - fraction * 0.42), height * 0.92))

def remove_detail(image, scale=0.18):
    # Smaller scale removes fine detail before the image is enlarged again.
    small = image.resize((int(image.width * scale), int(image.height * scale)))
    return small.resize(image.size)

def lower_light(image, factor=0.32):
    # factor below 1 darkens the image; 1 leaves brightness unchanged.
    return ImageEnhance.Brightness(image).enhance(factor)

def soften_edges(image, radius=8):
    # Larger radius makes boundaries and small details harder to see.
    return image.filter(ImageFilter.GaussianBlur(radius=radius))

def remove_colour(image):
    # Grayscale removes colour information while keeping the image in RGB mode.
    return ImageOps.grayscale(image).convert('RGB')


In [ ]:
# Run the five controlled transformations.
# Each item starts from original - the transformations do not stack.
transforms = {
    'Crop - context': crop_context(original),
    'Resize - detail': remove_detail(original),
    'Low light - contrast': lower_light(original),
    'Blur - edges': soften_edges(original),
    'Grayscale - colour': remove_colour(original),
}

# Compare one result at a time with the original.
for name, image in transforms.items():
    print(name)
    display(image)


## 3. Test a claim

Choose one transformation and write a prediction before you run it:

- What object or clue becomes harder to see?
- Would a detector have enough evidence to make the same claim?
- What would you record so another person could repeat your test?


In [ ]:
# Change ONE value, rerun this cell, and record both the value and your observation.
TEST_BLUR_RADIUS = 14
test = soften_edges(original, radius=TEST_BLUR_RADIUS)

print(f'Test: Gaussian blur radius = {TEST_BLUR_RADIUS}')
print('Original')
display(original)
print('Transformed result')
display(test)


## 4. Deliberately test failure

A clear image is an easy case. Test one difficult condition: low light, motion blur, a crop, a different angle, or an occlusion.

**Responsible-use prompt:** If this system makes a consequential decision, who checks the uncertain result, and what evidence do they need to see?
